# Price Estimator
This project uses an LLM to predict product prices from descriptions, benchmarks it against a simple average-price baseline, and lets users test predictions via a Gradio UI with real prices fetched from DummyJSON for comparison



In [ ]:
import os
import re
import sys
from pathlib import Path
from urllib.parse import quote_plus

import requests
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# Dynamically add the week6 folder to path so we can import pricer
possible_paths = [
    Path.cwd(),
    Path.cwd() / "week6",
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
for p in possible_paths:
    if (p / "pricer" / "items.py").exists():
        sys.path.insert(0, str(p))
        break

from pricer.items import Item
from pricer.evaluator import evaluate

# Load API key for OpenRouter
load_dotenv(override=True)
OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")
MODEL_NAME = "openai/gpt-5.4"

ai_client = (
    OpenAI(api_key=OPENROUTER_KEY, base_url="https://openrouter.ai/api/v1")
    if OPENROUTER_KEY
    else None
)

if not OPENROUTER_KEY:
    print("OPENROUTER_API_KEY not found. AI predictions will fall back to baseline.")

In [ ]:
# Load dataset from Hugging Face
DATASET_ID = "ed-donner/items_lite"
train_data, val_data, test_data = Item.from_hub(DATASET_ID)

print(f"Training examples: {len(train_data):,}")
print(f"Validation examples: {len(val_data):,}")
print(f"Test examples: {len(test_data):,}")

# Compute average price from training set (for baseline)
all_prices = [item.price for item in train_data]
AVG_PRICE = sum(all_prices) / len(all_prices)
print(f"Average training price: ${AVG_PRICE:.2f}")

In [ ]:
# Baseline predictor: always return the average price
def baseline_predictor(item):
    return AVG_PRICE

print("Evaluating baseline...")
evaluate(baseline_predictor, test_data, size=100)

In [ ]:
# LLM-based predictor
SYSTEM_INSTRUCTION = """You are an expert in product pricing.
Given a product description, output ONLY the estimated price in US dollars.
Do not include any extra text, currency symbols, or formatting; just the numeric value."""


def ai_estimator(item):
    if not ai_client:
        return AVG_PRICE

    # Use summary if available, otherwise fall back to title or string representation
    product_text = item.summary or item.title or str(item)
    product_text = (product_text or "")[:800]  # limit length

    try:
        response = ai_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_INSTRUCTION},
                {"role": "user", "content": f"Product description: {product_text}\n\nPrice:"},
            ],
            temperature=0.0,
            max_tokens=32,
        )
        raw_output = (response.choices[0].message.content or "").strip()
        # Extract the first number found (including decimals)
        match = re.search(r"(\d+(?:\.\d+)?)", raw_output)
        return float(match.group(1)) if match else AVG_PRICE
    except Exception as e:
        print(f"Error during API call: {e}")
        return AVG_PRICE


print("Evaluating AI estimator...")
evaluate(ai_estimator, test_data, size=100)

In [ ]:
# Gradio interface with external actual-price lookup
DUMMYJSON_SEARCH_URL = "https://dummyjson.com/products/search?q="


def fetch_actual_price_from_external_source(description: str):
    """Return (actual_price, product_title, product_url) from DummyJSON search."""
    query = quote_plus(description.strip())
    url = f"{DUMMYJSON_SEARCH_URL}{query}"

    try:
        response = requests.get(url, timeout=12)
        response.raise_for_status()
        payload = response.json()
        products = payload.get("products", [])
        if not products:
            return None, None, None

        product = products[0]
        actual_price = product.get("price")
        title = product.get("title", "Unknown product")
        product_id = product.get("id")
        product_url = f"https://dummyjson.com/products/{product_id}" if product_id else url

        if actual_price is None:
            return None, title, product_url
        return float(actual_price), title, product_url
    except Exception as exc:
        print(f"External lookup failed: {exc}")
        return None, None, None


def predict_price(description: str):
    if not description or not description.strip():
        return "Please enter a product description.", "No actual price lookup performed."

    # LLM prediction (fallbacks to AVG_PRICE if key is missing)
    dummy_item = Item(title="", category="", price=0.0, summary=description.strip())
    predicted = ai_estimator(dummy_item)
    predicted_text = f"Estimated price: ${predicted:.2f}"

    # External source actual price
    actual_price, matched_title, source_url = fetch_actual_price_from_external_source(description)

    if actual_price is None:
        actual_text = "Actual price: Not found from external source. Try a more specific product name."
        return predicted_text, actual_text

    error = abs(predicted - actual_price)
    direction = "higher" if predicted > actual_price else "lower"
    actual_text = (
        f"Actual price (external): ${actual_price:.2f}\n"
        f"Difference vs prediction: ${error:.2f} ({direction})\n"
        f"Matched product: {matched_title}\n"
        f"Source: {source_url}"
    )
    return predicted_text, actual_text


example_items = [
    ["Essence Mascara Lash Princess"],
    ["Eyeshadow Palette with Mirror"],
    ["Powder Canister"],
    ["Red Lipstick"],
    ["Red Nail Polish"],
]

with gr.Blocks(title="Price Estimator") as ui:
    gr.Markdown("## Product Description")
    description_input = gr.Textbox(
        label="Product description",
        lines=4,
        placeholder="e.g. Eyeshadow Palette with Mirror",
    )
    gr.Examples(examples=example_items, inputs=description_input)

    gr.Markdown("## Predicted Price")
    predicted_output = gr.Textbox(label="Predicted Price", lines=2)

    gr.Markdown("## Actual Prices")
    actual_output = gr.Textbox(label="Actual Price (External Source)", lines=5)

    estimate_button = gr.Button("Estimate and Compare")
    estimate_button.click(
        fn=predict_price,
        inputs=description_input,
        outputs=[predicted_output, actual_output],
    )

# prevent_thread_lock=True avoids blocking notebook execution checks
ui.launch(prevent_thread_lock=True)